In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42) # 재사용성

X = torch.tensor([ # 변수 선언 tensor
    [2.0, 4.0],  
    [8.0, 7.0],   
    [5.0, 3.0],   
    [9.0, 8.0],   
    [3.0, 6.0],  
    [7.0, 9.0],   
    [1.0, 2.0],   
    [6.0, 8.0],   
])

y = torch.tensor([
    [0.], [1.], [0.], [1.],
    [0.], [1.], [0.], [1.]
])

X = X / 10.0

print("=" * 45)
print("  1단계: 데이터 확인")
print("=" * 45)
print(f"  입력 shape : {X.shape}   (8명, 특성 2개)")
print(f"  정답 shape : {y.shape}  (8명, 출력 1개)")
print()

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU(),
            nn.Linear(4, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


model = SimpleNet()

total_params = sum(p.numel() for p in model.parameters())
print("=" * 45)
print("  2단계: 모델 구조")
print("=" * 45)
print(model)
print(f"\n  학습할 파라미터 수: {total_params}개")
print()

loss_fn   = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

print("=" * 45)
print("  3단계: 학습")
print("=" * 45)

for epoch in range(1000):

    pred = model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1:4d}/1000 | Loss: {loss.item():.4f}")

with torch.no_grad():
    predictions      = model(X)
    predicted_labels = (predictions > 0.5).float()
    accuracy         = (predicted_labels == y).float().mean()

    print("=" * 45)
    print("  4단계: 평가 & 예측")
    print("=" * 45)
    print(f"  학습 데이터 정확도: {accuracy.item() * 100:.1f}%")
    print()

    new_students = torch.tensor([
        [9.0, 9.0],
        [8.0, 8.0],
        [8.0, 7.0],
        [7.0, 8.0],
        [7.0, 6.0],
        [6.0, 7.0],
        [6.0, 5.0],
        [5.0, 6.0],
        [5.0, 5.0],
        [4.0, 5.0],
        [4.0, 4.0],
        [3.0, 4.0],
        [2.0, 3.0],
        [1.0, 8.0],
        [9.0, 1.0],
    ]) / 10.0

    labels = [
        "공부9h+수면9h",
        "공부8h+수면8h",
        "공부8h+수면7h",
        "공부7h+수면8h",
        "공부7h+수면6h",
        "공부6h+수면7h",
        "공부6h+수면5h",
        "공부5h+수면6h",
        "공부5h+수면5h",
        "공부4h+수면5h",
        "공부4h+수면4h",
        "공부3h+수면4h",
        "공부2h+수면3h",
        "공부1h+수면8h",
        "공부9h+수면1h",
    ]

    probs = model(new_students)

    print("  새 학생 예측 결과:")
    print(f"  {'케이스':<18} {'결과':<8} {'합격확률':>6}  {'불합격확률':>8}  막대")
    print("  " + "-" * 65)
    for label, prob in zip(labels, probs):
        p      = prob.item()
        fail_p = 1 - p                          # 불합격 확률
        result = "✅ 합격" if p > 0.5 else "❌ 불합격"
        bar    = "█" * int(p * 20)
        print(f"  {label:<18} {result:<8} {p:>6.2f}    {fail_p:>6.2f}      {bar}")

print()
print("  학습 완료!")

  1단계: 데이터 확인
  입력 shape : torch.Size([8, 2])   (8명, 특성 2개)
  정답 shape : torch.Size([8, 1])  (8명, 출력 1개)

  2단계: 모델 구조
SimpleNet(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=4, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

  학습할 파라미터 수: 37개

  3단계: 학습
  Epoch  100/1000 | Loss: 0.0002
  Epoch  200/1000 | Loss: 0.0001
  Epoch  300/1000 | Loss: 0.0001
  Epoch  400/1000 | Loss: 0.0000
  Epoch  500/1000 | Loss: 0.0000
  Epoch  600/1000 | Loss: 0.0000
  Epoch  700/1000 | Loss: 0.0000
  Epoch  800/1000 | Loss: 0.0000
  Epoch  900/1000 | Loss: 0.0000
  Epoch 1000/1000 | Loss: 0.0000

  4단계: 평가 & 예측
  학습 데이터 정확도: 100.0%

  새 학생 예측 결과:
  케이스                결과         합격확률     불합격확률  막대
  -----------------------------------------------------------------
  공부9h+수면9h          ✅ 합격       1.00      0.00      ████████████████████
  공부8h+수면8h   